# Reproduce Ground-Truth Figure Pipeline

This notebook is self-contained. The input validation, dot-bracket parsing, staging, manifest writing, and per-graph generation code are all in notebook cells.

**Notebook version:** cache-first by default. Existing PNGs are staged and displayed unless `force_regenerate = True`.


## Setup

In [ ]:
from pathlib import Path
import datetime as dt
import json
import os
import shutil
import subprocess
import sys

from IPython.display import Image, display

REPO = Path('/home/nagle/final_version')
MRNA_PLOT_DIR = REPO / 'mRNA_RBP' / 'outputs' / 'notebook_plots'
GT_COLLECTION_DIR = REPO / 'mRNA_RBP' / 'outputs' / 'ground_truth_collections'
PYTHON = sys.executable
SQUID_PYTHON = Path('/home/nagle/miniconda3/envs/squid/bin/python')
SQUID_PYTHON = str(SQUID_PYTHON if SQUID_PYTHON.exists() else PYTHON)
SQUID_LIB = '/home/nagle/miniconda3/envs/squid/lib'

env = os.environ.copy()
env.setdefault('PYTHONPATH', str(REPO))
env.setdefault('MPLCONFIGDIR', '/tmp/mplconfig')
env.setdefault('XDG_CACHE_HOME', '/tmp/xdg-cache')
ld_parts = [SQUID_LIB]
for part in os.environ.get('LD_LIBRARY_PATH', '').split(':'):
    if part and part != SQUID_LIB:
        ld_parts.append(part)
if '/usr/local/cuda-11.2/lib64' not in ld_parts:
    ld_parts.append('/usr/local/cuda-11.2/lib64')
env['LD_LIBRARY_PATH'] = ':'.join(ld_parts)



In [ ]:
NOTEBOOK_REVISION = 'ssm_delta_from_wt_2026_07_04'
print(f'Loaded notebook revision: {NOTEBOOK_REVISION}')


## Inputs

In [ ]:
ground_truth = 'synthetic'  # synthetic, residualbind, or deepsquid
sequence = 'AAAAAAAAGCGCAUGCUUGCAUGGCAUGCGCAAAAAAAAAA'
structure = '........((((((((.......))))))))..........'
motif_positions_text = '17-21'
out_dir = REPO / 'mRNA_RBP' / 'outputs' / 'reproducible_pipeline' / ground_truth
out_dir.mkdir(parents=True, exist_ok=True)

# Cache-first by default. Keep this False to stage/show existing PNGs.
# Set True only when you intentionally want to rerun the slow plot scripts.
force_regenerate = False



## Pipeline Helpers

These helpers validate inputs, parse RNAFold dot-bracket structures, stage outputs, and write the manifest.

In [ ]:
def parse_dot_bracket(dot_bracket):
    bracket_pairs = {'(': ')', '[': ']', '{': '}', '<': '>'}
    closing = {v: k for k, v in bracket_pairs.items()}
    stacks = {opener: [] for opener in bracket_pairs}
    pairs = []

    for idx, char in enumerate(dot_bracket.strip()):
        if char in bracket_pairs:
            stacks[char].append(idx)
        elif char in closing:
            opener = closing[char]
            if not stacks[opener]:
                raise ValueError(f'Unmatched closing bracket {char!r} at position {idx}')
            pairs.append((stacks[opener].pop(), idx))
        elif char == '.':
            continue
        else:
            raise ValueError(f'Unsupported dot-bracket character {char!r} at position {idx}')

    for opener, values in stacks.items():
        if values:
            raise ValueError(f'Unmatched opening bracket {opener!r} at position {values[-1]}')
    return sorted(pairs)


def parse_positions(text):
    if not text:
        return []
    out = []
    for part in text.split(','):
        part = part.strip()
        if not part:
            continue
        if '-' in part:
            start, end = [int(x) for x in part.split('-', 1)]
            out.extend(range(start, end + 1))
        else:
            out.append(int(part))
    return sorted(set(out))


def validate_inputs(sequence, structure):
    seq = sequence.strip().upper().replace('T', 'U')
    if not seq:
        raise ValueError('sequence cannot be empty')
    bad = sorted(set(seq) - set('ACGU'))
    if bad:
        raise ValueError(f'sequence contains non-RNA bases: {bad}')
    db = structure.strip()
    if len(seq) != len(db):
        raise ValueError(f'sequence length ({len(seq)}) must match structure length ({len(db)})')
    return seq, db, parse_dot_bracket(db)


def run(cmd):
    env['PYTHONPATH'] = str(REPO)
    env['LD_LIBRARY_PATH'] = ':'.join(ld_parts)
    print(' '.join(str(x) for x in cmd))
    completed = subprocess.run(
        [str(x) for x in cmd],
        cwd=REPO,
        env=env,
        check=False,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with return code {completed.returncode}: {cmd}')
    return completed.returncode


def cached_or_run(src, cmd, label):
    src = Path(src)
    if src.exists() and not force_regenerate:
        print(f'Using cached {label}: {src}')
    else:
        if src.exists():
            print(f'Regenerating {label}: {src}')
        else:
            print(f'No cached {label} found; generating {src}')
        run(cmd)
    if not src.exists():
        raise FileNotFoundError(src)
    return src


def stage(src, dst_name):
    src = Path(src)
    dst = out_dir / dst_name
    if not src.exists():
        raise FileNotFoundError(src)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    staged_outputs.append({'source': str(src), 'destination': str(dst), 'copied': True})
    return dst


def show(path):
    display(Image(filename=str(path)))


def write_manifest():
    manifest = {
        'created_at': dt.datetime.now(dt.timezone.utc).isoformat(),
        'repo': str(REPO),
        'inputs': {
            'ground_truth': ground_truth,
            'sequence': sequence_clean,
            'structure': structure_clean,
            'stem_pairs_zero_based': stem_pairs,
            'motif_positions_zero_based': motif_positions,
        },
        'staged_outputs': staged_outputs,
    }
    path = out_dir / 'pipeline_manifest.json'
    path.write_text(json.dumps(manifest, indent=2) + '\n')
    return path

## Validate Inputs And Start Manifest

In [ ]:
sequence_clean, structure_clean, stem_pairs = validate_inputs(sequence, structure)
motif_positions = parse_positions(motif_positions_text)
staged_outputs = []

input_summary = {
    'ground_truth': ground_truth,
    'sequence': sequence_clean,
    'structure': structure_clean,
    'stem_pairs_zero_based': stem_pairs,
    'motif_positions_zero_based': motif_positions,
}
(out_dir / 'pipeline_inputs.json').write_text(json.dumps(input_summary, indent=2) + '\n')
input_summary

## Ground Truth Coefficient Analysis

In [ ]:
if ground_truth == 'synthetic':
    coef_src = MRNA_PLOT_DIR / 'coefficients_map' / 'coefficients_groundtruth_vs_surrogate_synthetic.png'
    cached_or_run(coef_src, [SQUID_PYTHON, 'mRNA_RBP/plots/plot_coefficients.py'], 'coefficient analysis')
else:
    coef_src = MRNA_PLOT_DIR / 'coefficients_map' / 'coefficients_oracle_vs_surrogate_msi1.png'
    if not coef_src.exists():
        raise FileNotFoundError(coef_src)
    print(f'Using cached coefficient analysis: {coef_src}')

coef_dst = stage(coef_src, 'coefficient_analysis.png')
show(coef_dst)


## Random Library

In [ ]:
if ground_truth == 'synthetic':
    synthetic_random_sources = [
        MRNA_PLOT_DIR / 'rand_lib_dist_synthetic_high_wt.png',
        MRNA_PLOT_DIR / 'rand_lib_dist_synthetic_low_wt.png',
    ]
    if force_regenerate or any(not src.exists() for src in synthetic_random_sources):
        cmd = [SQUID_PYTHON, 'mRNA_RBP/plots/plot_synthetic_rand_region_distributions.py']
        if force_regenerate:
            cmd.append('--force')
        run(cmd)
    random_sources = synthetic_random_sources
elif ground_truth == 'residualbind':
    print('Using cached ResidualBind random-library figures from ground_truth_collections.')
    random_sources = [
        GT_COLLECTION_DIR / 'ResidualBind oracle MSI1' / 'figures' / 'rand_lib_dist_msi1_oracle_region_classes.png',
        GT_COLLECTION_DIR / 'ResidualBind oracle MSI1' / 'figures' / 'rand_lib_dist_msi1_oracle_region_classes_low_wt.png',
        GT_COLLECTION_DIR / 'ResidualBind oracle VTS1' / 'figures' / 'rand_lib_dist_vts1_oracle_region_classes.png',
        GT_COLLECTION_DIR / 'ResidualBind oracle VTS1' / 'figures' / 'rand_lib_dist_vts1_oracle_region_classes_low_wt.png',
    ]
elif ground_truth == 'deepsquid':
    print('Using cached deepSQUID random-library figures from ground_truth_collections.')
    random_sources = [
        GT_COLLECTION_DIR / 'deepSQUID MSI1' / 'figures' / 'rand_lib_dist_msi1_deepsquid.png',
        GT_COLLECTION_DIR / 'deepSQUID VTS1' / 'figures' / 'rand_lib_dist_vts1_deepsquid.png',
    ]
else:
    raise ValueError(f'Unsupported ground_truth: {ground_truth}')

if not random_sources:
    raise FileNotFoundError('No random-library figures found')

for src in random_sources:
    if not src.exists():
        raise FileNotFoundError(src)
    dst = stage(src, Path('random_library') / src.name)
    print(dst.name)
    show(dst)



## Evaluation Library Distributions

In [ ]:
lib_src = MRNA_PLOT_DIR / 'library_distributions.png'
cached_or_run(lib_src, [SQUID_PYTHON, 'mRNA_RBP/plots/plot_library_distributions.py'], 'evaluation library distributions')
lib_dist_dst = stage(lib_src, 'evaluation_library_distributions.png')
show(lib_dist_dst)


## Scatterplot Across Mutation Rate

In [ ]:
scatter_src = MRNA_PLOT_DIR / 'scatter_by_mutcount.png'
cached_or_run(
    scatter_src,
    [SQUID_PYTHON, 'mRNA_RBP/plots/plot_scatter_by_mutcount.py', '--instance', '0', '--lib_size', '20000'],
    'scatterplot across mutation rate',
)
scatter_dst = stage(scatter_src, 'scatter_by_mutcount.png')
show(scatter_dst)


## Library Size

In [ ]:
rho_src = MRNA_PLOT_DIR / 'rho_vs_libsize_type3.png'
if force_regenerate or not rho_src.exists():
    run([SQUID_PYTHON, 'mRNA_RBP/lib_size_spearman.py', '--out_json', 'mRNA_RBP/outputs/lib_size_spearman_results_type3.json', '--recompute_saturated', '--saturated_only', '--gt_keys', 'additive', 'additive_pairwise', 'nonlin_additive', 'nonlin_additive_pairwise'])
    run([SQUID_PYTHON, 'mRNA_RBP/plots/plot_rho_vs_libsize_type3.py'])
else:
    print(f'[cache] using existing library-size plot: {rho_src}')
rho_dst = stage(rho_src, 'rho_vs_libsize_type3.png')
show(rho_dst)


## Spearman vs RMSE Model Misspecification

In [ ]:
model_src = MRNA_PLOT_DIR / 'model_comparison_bar_type3.png'
cached_or_run(model_src, [SQUID_PYTHON, 'mRNA_RBP/plots/bar_surrogate_models_type3.py'], 'model misspecification plot')
model_dst = stage(model_src, 'model_comparison_type3.png')
show(model_dst)


## Cross-Mutation-Rate Heatmap

In [ ]:
cross_src = MRNA_PLOT_DIR / 'synthetic_gt_cross_mutrate_heatmap.png'
cached_or_run(
    cross_src,
    [
        SQUID_PYTHON,
        'mRNA_RBP/plots/plot_cross_mutrate.py',
        '--out_prefix', 'synthetic_gt_',
        '--out_base', 'mRNA_RBP/outputs',
    ],
    'cross-mutation-rate heatmap',
)
cross_dst = stage(cross_src, 'cross_mutrate_heatmap.png')
show(cross_dst)


## Final Manifest

In [ ]:
manifest_path = write_manifest()
json.loads(manifest_path.read_text())['inputs']